# 03 — Baseline Models: SARIMA & Prophet

Trains and evaluates SARIMA and Prophet on the Moroccan load data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
from pathlib import Path

ROOT = Path('..')

In [ ]:
from src.data.fetch_data import fetch_entsoe_load, fetch_openmeteo_weather
from src.data.preprocess import build_features, train_test_split_temporal

load    = fetch_entsoe_load()
weather = fetch_openmeteo_weather()
df      = build_features(load, weather)
train, test = train_test_split_temporal(df)

HORIZON = 7 * 24  # 7 days
print(f'Train: {len(train)}h  |  Test: {len(test)}h  |  Horizon: {HORIZON}h')

## 1. SARIMA

In [ ]:
from src.models.baseline import SARIMAForecaster
from src.evaluation.metrics import evaluate_forecasts

sarima = SARIMAForecaster()
sarima.fit(train['load_MW'])
sarima_preds = sarima.predict(HORIZON)

sarima_metrics = evaluate_forecasts(
    test['load_MW'].iloc[:HORIZON],
    sarima_preds['forecast'],
    lower_80=sarima_preds['lower_80'],
    upper_80=sarima_preds['upper_80'],
)
print('SARIMA metrics:', sarima_metrics)

## 2. Prophet

In [ ]:
from src.models.baseline import ProphetForecaster

prophet = ProphetForecaster()
prophet.fit(train['load_MW'])
prophet_preds = prophet.predict(HORIZON)

prophet_metrics = evaluate_forecasts(
    test['load_MW'].iloc[:HORIZON],
    prophet_preds['forecast'],
    lower_80=prophet_preds['lower_80'],
    upper_80=prophet_preds['upper_80'],
)
print('Prophet metrics:', prophet_metrics)

## 3. Visualise forecasts

In [ ]:
from src.visualization.plots import forecast_ribbon_plotly

fig = forecast_ribbon_plotly(df['load_MW'], sarima_preds, title='SARIMA Forecast')
fig.show()

fig2 = forecast_ribbon_plotly(df['load_MW'], prophet_preds, title='Prophet Forecast')
fig2.show()

## 4. Summary table

In [ ]:
summary = pd.DataFrame([
    {'model': 'SARIMA',  **sarima_metrics},
    {'model': 'Prophet', **prophet_metrics},
])
summary